# 05 - Blocking: `ml.bus_matching_candidates`

Fifth notebook: Section 2 of the plan. Joins the two signature tables
on `(date, cell_x, cell_y, time_bucket)` -- a hash/merge join over
integers, no geometry -- aggregates to `(date, bus_id, device_id)`
overlap counts, keeps the top 10 devices per bus-date by overlap, and
unions in every dictionary-sourced pair for that bus regardless of
rank. Everything else never gets scored.

**Confirmed live before committing to the full build**: one date alone
(2023-11-01, 1,682 buses x 1,495 devices) produces **1.17M** raw
`(bus_id, device_id)` pairs with *any* nonzero cell/bucket overlap --
the grid+bucket signature alone is not very selective over a full day
(routes physically overlap all over the network, and a 15-minute bucket
is coarse across 18+ operating hours), confirmed at 15.5s for that one
date via `EXPLAIN ANALYZE`. This is expected and is exactly why the
plan doesn't stop at "nonzero overlap" -- the top-10-per-bus-date cut
below is what actually does the blocking; raw overlap counts are never
persisted on their own.


In [1]:
import os
import time
from pathlib import Path

import psycopg

In [2]:
_root = Path.cwd()
while not (_root / "pyproject.toml").exists():
    _root = _root.parent
os.chdir(_root)
os.environ.setdefault("RAW_DATA_ROOT", str(_root))

'/home/victor/repos/opa-database'

In [3]:
from opa_database.config import settings

TOP_N_PER_BUS_DATE = 10

conn = psycopg.connect(settings.db_dsn)
conn.execute("CREATE SCHEMA IF NOT EXISTS ml;")
conn.commit()
print("ml schema ready")

ml schema ready


## Build

`bus_dates` is every `(bus_id, trip_date)` with at least one valid trip
-- the actual unit of prediction, not just whatever happens to have a
GTFS-matched signature. `top_blocking` keeps the best
`TOP_N_PER_BUS_DATE` devices per bus-date by overlap count.
`dictionary_candidates` cross-joins every `bus_matching_candidate_pairs`
row onto every date that bus actually ran, since the dictionary itself
carries no date. A `FULL OUTER JOIN` merges the two so a pair that's
both dictionary-sourced *and* blocking-ranked keeps its overlap count
instead of appearing twice.


In [4]:
start = time.monotonic()
conn.execute("DROP TABLE IF EXISTS ml.bus_matching_candidates;")
conn.execute(
    """
    CREATE TABLE ml.bus_matching_candidates AS
    WITH overlap AS (
        SELECT bs.date, bs.bus_id, ds.device_id, count(*) AS overlap_count
        FROM ml.bus_matching_bus_signatures bs
        JOIN ml.bus_matching_device_signatures ds
          ON ds.date = bs.date AND ds.cell_x = bs.cell_x
         AND ds.cell_y = bs.cell_y AND ds.time_bucket = bs.time_bucket
        GROUP BY bs.date, bs.bus_id, ds.device_id
    ),
    ranked AS (
        SELECT *,
            row_number() OVER (
                PARTITION BY date, bus_id ORDER BY overlap_count DESC
            ) AS rnk
        FROM overlap
    ),
    top_blocking AS (
        SELECT date, bus_id, device_id, overlap_count
        FROM ranked
        WHERE rnk <= %(top_n)s
    ),
    bus_dates AS (
        SELECT DISTINCT bus_id, trip_date AS date
        FROM ml.trip_validity_final
        WHERE is_valid
    ),
    dictionary_pairs_distinct AS (
        SELECT DISTINCT bus_id, device_id FROM ml.bus_matching_candidate_pairs
    ),
    dictionary_candidates AS (
        SELECT bd.date, bd.bus_id, cp.device_id
        FROM bus_dates bd
        JOIN dictionary_pairs_distinct cp ON cp.bus_id = bd.bus_id
    )
    SELECT
        coalesce(tb.date, dc.date) AS date,
        coalesce(tb.bus_id, dc.bus_id) AS bus_id,
        coalesce(tb.device_id, dc.device_id) AS device_id,
        tb.overlap_count,
        (dc.device_id IS NOT NULL) AS from_dictionary,
        (tb.device_id IS NOT NULL) AS from_blocking
    FROM top_blocking tb
    FULL OUTER JOIN dictionary_candidates dc
      ON dc.date = tb.date AND dc.bus_id = tb.bus_id AND dc.device_id = tb.device_id;
    """,
    {"top_n": TOP_N_PER_BUS_DATE},
)
conn.commit()
print(f"ml.bus_matching_candidates built in {time.monotonic() - start:.1f}s")

ml.bus_matching_candidates built in 588.0s


In [5]:
start = time.monotonic()
conn.execute("CREATE INDEX ON ml.bus_matching_candidates (date, bus_id);")
conn.execute("CREATE INDEX ON ml.bus_matching_candidates (date, device_id);")
conn.execute("ANALYZE ml.bus_matching_candidates;")
conn.commit()
print(f"indexes built in {time.monotonic() - start:.1f}s")

indexes built in 0.5s


## Sanity checks

In [6]:
with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM ml.bus_matching_candidates;")
    print("total candidate rows:", cur.fetchone())

    cur.execute(
        "SELECT count(*), count(*) FILTER (WHERE from_dictionary), "
        "count(*) FILTER (WHERE from_blocking), "
        "count(*) FILTER (WHERE from_dictionary AND from_blocking) "
        "FROM ml.bus_matching_candidates;"
    )
    print("total / from_dictionary / from_blocking / both:", cur.fetchone())

    cur.execute(
        """
        SELECT count(*), avg(n)::numeric(10,2), min(n), max(n) FROM (
            SELECT date, bus_id, count(*) AS n
            FROM ml.bus_matching_candidates GROUP BY date, bus_id
        ) x;
        """
    )
    print(
        "bus-dates covered / avg / min / max candidates per bus-date:", cur.fetchone()
    )

    cur.execute(
        """
        SELECT count(*) FROM (
            SELECT DISTINCT bus_id, trip_date FROM ml.trip_validity_final WHERE is_valid
        ) bd
        WHERE NOT EXISTS (
            SELECT 1 FROM ml.bus_matching_candidates c
            WHERE c.bus_id = bd.bus_id AND c.date = bd.trip_date
        );
        """
    )
    print("valid bus-dates with zero candidates at all:", cur.fetchone())

    cur.execute(
        """
        SELECT count(*) FROM ml.bus_matching_candidate_pairs cp
        WHERE NOT EXISTS (
            SELECT 1 FROM ml.bus_matching_candidates c
            WHERE c.bus_id = cp.bus_id AND c.device_id = cp.device_id
        );
        """
    )
    print(
        "dictionary pairs missing from candidates entirely (should be 0):",
        cur.fetchone(),
    )

total candidate rows: (407517,)
total / from_dictionary / from_blocking / both: (407517, 40265, 399212, 31960)
bus-dates covered / avg / min / max candidates per bus-date: (40007, Decimal('10.19'), 1, 13)


valid bus-dates with zero candidates at all: (1325,)
dictionary pairs missing from candidates entirely (should be 0): (0,)
